### Restart and Run All Cells

In [1]:
import pandas as pd
from sqlalchemy import create_engine
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
year = 2022
quarter = 4

In [2]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',    
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%'
              }

In [3]:
sql = """
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2022 AND quarter <= 4) 
OR (year = 2022-1 AND quarter >= 4+1)
ORDER BY year DESC, quarter DESC


In [4]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
1,ADVANC,8088,10,"26,011,284",4
10,AOT,8088,10,"-11,087,867",4
14,ASP,8088,10,"479,274",4
20,BAY,8088,10,"30,712,985",4
21,BBL,8088,10,"29,305,592",4


In [5]:
sql = """
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s-1 AND quarter <= %s) 
OR (year = %s-2 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC
"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2022-1 AND quarter <= 4) 
OR (year = 2022-2 AND quarter >= 4+1)
ORDER BY year DESC, quarter DESC



In [6]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,ACE,8084,10,"1,505,681",4
1,ADVANC,8084,10,"26,922,146",4
2,AEONTS,8084,10,"3,553,041",4
3,AH,8084,10,"1,023,968",4
4,AIE,8084,10,"423,622",4


In [7]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,ADVANC,2022,Q4,"26,011,284","26,922,146","-910,862",-3.38%
1,AOT,2022,Q4,"-11,087,867","-16,322,014","5,234,147",32.07%
2,ASP,2022,Q4,"479,274","978,355","-499,081",-51.01%
3,BAY,2022,Q4,"30,712,985","33,794,188","-3,081,203",-9.12%
4,BBL,2022,Q4,"29,305,592","26,507,039","2,798,553",10.56%


In [8]:
sql = '''
DELETE FROM yr_profits 
WHERE year = %s AND quarter = "Q%s"'''
sql = sql % (year, quarter)
print(sql)

rp = conlt.execute(sql)
rp.rowcount


DELETE FROM yr_profits 
WHERE year = 2022 AND quarter = "Q4"


20

In [9]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
rcds = df_ins.values.tolist()
len(rcds)

32

In [10]:
#for rcd in rcds:
#    print(rcd)

In [11]:
sql = """
INSERT INTO yr_profits (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id) 
VALUES (?, ?, ?, ?, ?, ?, ?, ?)"""
print(sql)


INSERT INTO yr_profits (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id) 
VALUES (?, ?, ?, ?, ?, ?, ?, ?)


In [12]:
for rcd in rcds:
    conlt.execute(sql, rcd)

### End of loop

In [13]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,ADVANC,2022,Q4,"26,011,284","26,922,146","-910,862",-3.38%
2,ASP,2022,Q4,"479,274","978,355","-499,081",-51.01%
3,BAY,2022,Q4,"30,712,985","33,794,188","-3,081,203",-9.12%
4,BBL,2022,Q4,"29,305,592","26,507,039","2,798,553",10.56%
6,DTAC,2022,Q4,"3,119,157","3,355,933","-236,776",-7.06%


In [14]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,ADVANC,2022,Q4,"26,011,284","26,922,146","-910,862",-3.38%
2,ASP,2022,Q4,"479,274","978,355","-499,081",-51.01%
3,BAY,2022,Q4,"30,712,985","33,794,188","-3,081,203",-9.12%
4,BBL,2022,Q4,"29,305,592","26,507,039","2,798,553",10.56%
5,COTTO,2022,Q4,"-227,793","583,604","-811,397",-139.03%


In [15]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
1,AOT,2022,Q4,"-11,087,867","-16,322,014","5,234,147",32.07%
4,BBL,2022,Q4,"29,305,592","26,507,039","2,798,553",10.56%
7,GGC,2022,Q4,"953,296","330,219","623,077",188.69%
15,KKP,2022,Q4,"7,602,096","6,318,052","1,284,044",20.32%
16,KTB,2022,Q4,"33,697,736","21,588,288","12,109,448",56.09%


In [16]:
final_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[final_criteria, cols].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
29,TOP,2022,Q4,"32,668,135","12,578,026","20,090,109",159.72%
20,OISHI,2022,Q4,"1,197,453","546,690","650,763",119.04%
22,PTTEP,2022,Q4,"70,901,335","38,863,595","32,037,740",82.44%
19,MC,2022,Q4,"740,276","445,354","294,922",66.22%
27,TFFIF,2022,Q4,"1,851,409","1,125,531","725,878",64.49%


In [17]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
4,BBL,2022,Q4,"29,305,592","26,507,039","2,798,553",10.56%
15,KKP,2022,Q4,"7,602,096","6,318,052","1,284,044",20.32%
16,KTB,2022,Q4,"33,697,736","21,588,288","12,109,448",56.09%
17,KTC,2022,Q4,"7,079,398","5,878,692","1,200,706",20.42%
18,LHFG,2022,Q4,"1,578,755","1,383,721","195,034",14.09%


In [18]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
4,BBL,2022,Q4,"29,305,592","26,507,039","2,798,553",10.56%
15,KKP,2022,Q4,"7,602,096","6,318,052","1,284,044",20.32%
16,KTB,2022,Q4,"33,697,736","21,588,288","12,109,448",56.09%
17,KTC,2022,Q4,"7,079,398","5,878,692","1,200,706",20.42%
18,LHFG,2022,Q4,"1,578,755","1,383,721","195,034",14.09%
